# Notebook 05 — Data Quality Fix (v2)

**Why this notebook exists:** The first run showed the fine-tuned BiEncoder
barely beating its untrained baseline (MRR went from 0.083 to 0.096). The
root cause was bad training data — chunks started mid-sentence and the
first-sentence-as-query trick produced garbage like "templates. So a neural
network is like multinomial logistic regression..."

This notebook fixes both problems:
1. Re-chunk the PDF respecting **sentence boundaries** (`src/chunker_v2.py`).
2. Generate training queries with **an LLM** instead of extracting them
   (`src/query_generator.py`).
3. Build new train/val splits and save them.

After running this notebook, re-run `02_training.ipynb` with the new pair
files. You should see MRR@10 jump from ~0.10 to ~0.5+ (still under BM25's
1.0 on this eval set, but a clear, defensible gain).

---
**Choose your query-generation mode:**
- **A — Anthropic API** (recommended): ~$0.50, ~10 min for the whole corpus.
- **B — OpenAI API**: ~$0.40, ~10 min.
- **C — Manual batches**: free, ~30 min of pasting prompts into Claude.ai / ChatGPT.

## Setup

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Neural_Search_Engine'
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
    !pip install -q pymupdf anthropic openai
else:
    PROJECT_ROOT = os.path.abspath('..')
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

print('Working directory:', os.getcwd())

---
## Step 1 — Re-chunk the PDF with sentence boundaries

In [ ]:
import json
from src.utils import extract_clean_text_from_pdf
from src.chunker_v2 import SentenceAwareChunker, split_into_sentences

PDF_PATH = 'data/jurafsky_martin.pdf'
OUTPUT_PATH = 'data/processed/jurafsky_chunks_v2.json'

# Step 1a: extract clean text from the PDF (reuses your existing utility)
raw_text = extract_clean_text_from_pdf(PDF_PATH)
print(f'Extracted {len(raw_text):,} characters')

# Step 1b: chunk with sentence-aware logic
chunker = SentenceAwareChunker(target_words=220, max_words=300, overlap_sentences=2)
chunks_v2 = chunker.chunk(raw_text)

# Step 1c: save
os.makedirs('data/processed', exist_ok=True)
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(chunks_v2, f, indent=2, ensure_ascii=False)

print(f'\nWrote {len(chunks_v2)} chunks -> {OUTPUT_PATH}')
wcs = [c['word_count'] for c in chunks_v2]
print(f'Word count: min={min(wcs)}, max={max(wcs)}, avg={sum(wcs)/len(wcs):.0f}')

In [ ]:
# Verify the fix — every chunk should now start with a real sentence
print('First sentence of first 5 chunks (compare to your old jurafsky_chunks.json):')
for c in chunks_v2[:5]:
    first = split_into_sentences(c['content'])[0]
    safe = first.encode('ascii', 'replace').decode()
    print(f"  [{c['id']}] {safe[:140]}")

---
## Step 2 — Pick ONE query-generation mode
Run one of the next three cells (A, B, or C), then continue to Step 3.

### Mode A — Anthropic Claude API (recommended)

In [ ]:
# Skip this cell if you're using mode B or C.
import os
from src.query_generator import AnthropicQueryGenerator

# In Colab: store your key in a secret named ANTHROPIC_API_KEY
if IN_COLAB:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

# Sanity test: generate queries for 1 chunk first
gen = AnthropicQueryGenerator(model='claude-haiku-4-5', n_queries=2)
sample_queries = gen.generate_for_chunk(chunks_v2[10])
print(f"Sample queries for {chunks_v2[10]['id']}:")
for q in sample_queries:
    print(f'  - {q}')

# If that looks good, generate for the whole corpus
# (Comment this out for testing; uncomment when ready)
# pairs = gen.generate_pairs(chunks_v2, output_path='data/processed/llm_pairs.json')

### Mode B — OpenAI API

In [ ]:
# Skip if using mode A or C.
# from src.query_generator import OpenAIQueryGenerator
# if IN_COLAB:
#     from google.colab import userdata
#     os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
# gen = OpenAIQueryGenerator(model='gpt-4o-mini', n_queries=2)
# sample_queries = gen.generate_for_chunk(chunks_v2[10])
# print(sample_queries)
# pairs = gen.generate_pairs(chunks_v2, output_path='data/processed/llm_pairs.json')

### Mode C — Manual / batch mode (no API key)

Two passes:
1. Run cell C1 → writes prompt files to `data/prompts/`. Paste each one
   into Claude.ai / ChatGPT / Gemini and save the response as
   `data/prompts/response_NNN.json`.
2. Run cell C2 → ingests the responses into `data/processed/llm_pairs.json`.

In [ ]:
# Cell C1 — write prompt files (one per batch of 20 chunks)
# from src.query_generator import ManualBatchGenerator
# manual = ManualBatchGenerator(n_queries=2)
# manual.batch_prompts(chunks_v2, out_dir='data/prompts', batch_size=20)

In [ ]:
# Cell C2 — after you've collected all response_NNN.json files
# manual.ingest_responses(
#     responses_dir='data/prompts',
#     chunks=chunks_v2,
#     output_path='data/processed/llm_pairs.json',
# )

---
## Step 3 — Inspect the generated queries

In [ ]:
with open('data/processed/llm_pairs.json', encoding='utf-8') as f:
    llm_pairs = json.load(f)

print(f'Loaded {len(llm_pairs)} (query, positive) pairs')
print(f'Unique chunks covered: {len(set(p["positive_id"] for p in llm_pairs))}')
print(f'Avg queries per chunk: {len(llm_pairs) / len(set(p["positive_id"] for p in llm_pairs)):.1f}')

print('\n--- Sample LLM-generated queries (compare to first-sentence garbage) ---\n')
for p in llm_pairs[::200][:5]:
    print(f"Q: {p['query']}")
    print(f"   matches [{p['positive_id']}]: {p['positive_text'][:120]}...\n")

---
## Step 4 — Add random negatives and write train/val pairs

In [ ]:
import random
from src.query_generator import add_random_negatives
from src.dataset import save_pairs

# 4a: enrich each (query, positive) pair with a random distant negative
triplets = add_random_negatives(llm_pairs, chunks_v2, min_distance=20, seed=42)
print(f'Enriched to {len(triplets)} (query, positive, negative) triplets')

# 4b: shuffle and split 90/10 train/val
rng = random.Random(42)
rng.shuffle(triplets)
cut = int(len(triplets) * 0.9)
train_pairs = triplets[:cut]
val_pairs   = triplets[cut:]

# 4c: save with the SAME filenames that 02_training.ipynb expects
save_pairs(train_pairs, 'data/processed/train_pairs.json')
save_pairs(val_pairs,   'data/processed/val_pairs.json')

print(f'\nTrain pairs: {len(train_pairs)}')
print(f'Val pairs:   {len(val_pairs)}')
print(f'Test queries (unchanged): 10 in data/evaluation_set.csv')

---
## Step 5 — Update the corpus pointer

`02_training.ipynb` reads from `data/processed/jurafsky_chunks.json`. We swap
in the v2 chunks by re-pointing that file.

In [ ]:
import shutil

# Back up the old file before replacing
if os.path.exists('data/processed/jurafsky_chunks.json'):
    shutil.copy('data/processed/jurafsky_chunks.json', 'data/processed/jurafsky_chunks_v1_backup.json')
    print('Backed up old chunks to jurafsky_chunks_v1_backup.json')

shutil.copy('data/processed/jurafsky_chunks_v2.json', 'data/processed/jurafsky_chunks.json')
print('Promoted v2 chunks to the canonical jurafsky_chunks.json')

with open('data/processed/jurafsky_chunks.json', encoding='utf-8') as f:
    final = json.load(f)
print(f'Canonical corpus now has {len(final)} chunks.')

---
## Next step

Re-run **`02_training.ipynb`** with the same hyperparameters. Then rerun
**`03_evaluation.ipynb`**. Expected delta: MRR@10 jumps from ~0.10 (the
broken run) to **0.5+** on the existing 10-query eval set.

**Also strongly recommended:** add 10 more eval queries to
`data/evaluation_set.csv` that are deliberately **paraphrased** — using
synonyms and rephrased phrasing instead of textbook vocabulary. That's where
the BiEncoder will start beating BM25, giving you the contrast your report
needs.